## Importing Libraries and Loading our Models.

In this section, we import all required libraries and load the pretrained Stack and Kaggle BERT models along with their corresponding tokenizers.

In [7]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from utils import tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model_path_stack = "nemoonyte/stack-bert-model"
model_path_kaggle = "nemoonyte/kaggle-bert-model"

tokenizer_stack = BertTokenizer.from_pretrained(model_path_stack)
tokenizer_kaggle = BertTokenizer.from_pretrained(model_path_kaggle)

model_stack = BertForSequenceClassification.from_pretrained(model_path_stack).to(device)
model_kaggle = BertForSequenceClassification.from_pretrained(model_path_kaggle).to(device)

model_stack.eval()
model_kaggle.eval()

cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## Loading and Inspect Test Datasets

We load both the Stack Overflow and Kaggle test datasets and verify their structure.

In [8]:

# now we load both test datasets

test_df_stack = pd.read_csv("dataset/stack_test_clean.csv")
test_df_kaggle = pd.read_csv("dataset/kaggle_test_clean.csv")

print("Stack:", test_df_stack.shape)
print(test_df_stack.columns.tolist())

print("\nKaggle:", test_df_kaggle.shape)
print(test_df_kaggle.columns.tolist())

Stack: (1600, 3)
['source', 'raw_text', 'priority_label']

Kaggle: (1000, 3)
['source', 'raw_text', 'priority_label']


## Defining Dataset Class and Evaluation Function

We define a PyTorch dataset class and an evaluation function to run inference and compute predictions.

In [9]:
label_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2}

text_column = "raw_text"
label_column = "priority_label"

class QueryDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item

def evaluate_model(model, dataloader):
    all_preds, all_true = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_true.extend(labels.cpu().tolist())

    return all_true, all_preds

print(label_map)

{'Low': 0, 'Medium': 1, 'High': 2}


## Evaluating Kaggle Model on Kaggle Test Set

This evaluates how well the Kaggle-trained model performs on its own dataset.

In [10]:
# now we test the kaggle model on the kaggle test set

# first we tokenize the text so the model can understand it


kaggle_encodings, kaggle_labels = tokenize(
    test_df_kaggle,
    text_column=text_column,
    tokenizer=tokenizer_kaggle,
    labels=label_column
)
# convert labels to numbers (same as training)
kaggle_labels_encoded = kaggle_labels.map(label_map)

kaggle_loader = DataLoader(
    QueryDataset(kaggle_encodings, kaggle_labels_encoded),
    batch_size=16,
    shuffle=False
)

kaggle_true, kaggle_preds = evaluate_model(model_kaggle, kaggle_loader)

print("Kaggle Model → Kaggle Test Set")
print("Accuracy:", accuracy_score(kaggle_true, kaggle_preds))
print(classification_report(kaggle_true, kaggle_preds, target_names=["Low","Medium","High"]))
print(confusion_matrix(kaggle_true, kaggle_preds))

Kaggle Model → Kaggle Test Set
Accuracy: 1.0
              precision    recall  f1-score   support

         Low       1.00      1.00      1.00       360
      Medium       1.00      1.00      1.00       311
        High       1.00      1.00      1.00       329

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

[[360   0   0]
 [  0 311   0]
 [  0   0 329]]


## Evaluating Kaggle Model on Stack Test Set

This tests how well the Kaggle model generalizes to Stack Overflow data.

In [11]:
stack_enc_kaggle, stack_lab_kaggle = tokenize(
    test_df_stack,
    text_column=text_column,
    tokenizer=tokenizer_kaggle,
    labels=label_column
)

stack_lab_kaggle_encoded = stack_lab_kaggle.map(label_map)

stack_loader_kaggle = DataLoader(
    QueryDataset(stack_enc_kaggle, stack_lab_kaggle_encoded),
    batch_size=16,
    shuffle=False
)

kaggle_on_stack_true, kaggle_on_stack_preds = evaluate_model(model_kaggle, stack_loader_kaggle)

print("Kaggle Model → Stack Test Set")
print("Accuracy:", accuracy_score(kaggle_on_stack_true, kaggle_on_stack_preds))

Kaggle Model → Stack Test Set
Accuracy: 0.48375


## Evaluating Stack Model on Stack Test Set

This evaluates how well the Stack-trained model performs on its own dataset.

In [12]:
stack_encodings, stack_labels = tokenize(
    test_df_stack,
    text_column=text_column,
    tokenizer=tokenizer_stack,
    labels=label_column
)

stack_labels_encoded = stack_labels.map(label_map)

stack_loader = DataLoader(
    QueryDataset(stack_encodings, stack_labels_encoded),
    batch_size=16,
    shuffle=False
)

stack_true, stack_preds = evaluate_model(model_stack, stack_loader)

print("Stack Model → Stack Test Set")
print("Accuracy:", accuracy_score(stack_true, stack_preds))
print(classification_report(stack_true, stack_preds, target_names=["Low","Medium","High"]))
print(confusion_matrix(stack_true, stack_preds))

Stack Model → Stack Test Set
Accuracy: 0.921875
              precision    recall  f1-score   support

         Low       0.87      0.97      0.92       611
      Medium       0.96      0.93      0.95       923
        High       1.00      0.29      0.45        66

    accuracy                           0.92      1600
   macro avg       0.94      0.73      0.77      1600
weighted avg       0.93      0.92      0.92      1600

[[595  16   0]
 [ 62 861   0]
 [ 25  22  19]]


## Evaluating Stack Model on Kaggle Test Set

This tests how well the Stack model generalizes to Kaggle data.

In [13]:
kaggle_enc_stack, kaggle_lab_stack = tokenize(
    test_df_kaggle,
    text_column=text_column,
    tokenizer=tokenizer_stack,
    labels=label_column
)

kaggle_lab_stack_encoded = kaggle_lab_stack.map(label_map)

kaggle_loader_stack = DataLoader(
    QueryDataset(kaggle_enc_stack, kaggle_lab_stack_encoded),
    batch_size=16,
    shuffle=False
)

stack_on_kaggle_true, stack_on_kaggle_preds = evaluate_model(model_stack, kaggle_loader_stack)

print("Stack Model → Kaggle Test Set")
print("Accuracy:", accuracy_score(stack_on_kaggle_true, stack_on_kaggle_preds))

Stack Model → Kaggle Test Set
Accuracy: 0.35
